In [1]:
import oracledb
import pandas as pd
try:
    conn = oracledb.connect(
        user = 'hr',
        password = 'basededatos',
        dsn = 'localhost:1521/XE'
    )
    cursor = conn.cursor()
    cursor.callproc('dbms_output.enable')
except oracledb.Error as e:
    print(e)

crear un procedimiento que llame a una funcion f_localidad y muestre el nombre del empleado y el valor de f_localidad

f_localidad dado un id del empleado da la localidad del empleado

In [11]:
try:
    cursor.execute(
    """
    CREATE OR REPLACE FUNCTION f_localidad
    (p_emp_id IN employees.employee_id%TYPE)
    RETURN VARCHAR2
    IS
        v_localidad locations.city%TYPE;
    BEGIN
        SELECT l.city
        INTO v_localidad
        FROM employees e
        JOIN departments d ON e.department_id = d.department_id
        JOIN locations l ON d.location_id = l.location_id
        WHERE e.employee_id = p_emp_id;
        RETURN v_localidad;
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            RETURN 'Sin localidad';
    END f_localidad;
    """
    )

    cursor.execute(
    """
    CREATE OR REPLACE PROCEDURE pro_localidad
    IS
        CURSOR empleados IS
            SELECT first_name || ' ' || last_name, f_localidad(employee_id)
            FROM employees;
        v_nombre VARCHAR2(50);
        v_localidad locations.city%TYPE;
    BEGIN
        OPEN empleados;
        LOOP
            FETCH empleados INTO v_nombre, v_localidad;
            EXIT WHEN empleados%NOTFOUND;
            DBMS_OUTPUT.PUT_LINE('Nombre: ' || v_nombre || ', Localidad: ' || v_localidad);
        END LOOP;
        CLOSE empleados;
    END pro_localidad;
    """
    )

    cursor.execute(
    """
    BEGIN
        pro_localidad;
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc('dbms_output.get_line', (line, status))
        if status.getvalue() != 0:
            break    
        print(line.getvalue())

except oracledb.Error as e:
    print(e)

Nombre: Steven King, Localidad: Seattle
Nombre: Neena Kochhar, Localidad: Seattle
Nombre: Lex De Haan, Localidad: Seattle
Nombre: Alexander Hunold, Localidad: Southlake
Nombre: Bruce Ernst, Localidad: Southlake
Nombre: David Austin, Localidad: Southlake
Nombre: Valli Pataballa, Localidad: Southlake
Nombre: Diana Lorentz, Localidad: Southlake
Nombre: Nancy Greenberg, Localidad: Seattle
Nombre: Daniel Faviet, Localidad: Seattle
Nombre: John Chen, Localidad: Seattle
Nombre: Ismael Sciarra, Localidad: Seattle
Nombre: Jose Manuel Urman, Localidad: Seattle
Nombre: Luis Popp, Localidad: Seattle
Nombre: Den Raphaely, Localidad: Seattle
Nombre: Alexander Khoo, Localidad: Seattle
Nombre: Shelli Baida, Localidad: Seattle
Nombre: Sigal Tobias, Localidad: Seattle
Nombre: Guy Himuro, Localidad: Seattle
Nombre: Karen Colmenares, Localidad: Seattle
Nombre: Matthew Weiss, Localidad: South San Francisco
Nombre: Adam Fripp, Localidad: South San Francisco
Nombre: Payam Kaufling, Localidad: South San Franc

In [12]:
import oracledb
import pandas as pd
try:
    conn = oracledb.connect(
        user = 'hr',
        password = 'basededatos',
        dsn = 'localhost:1521/XE'
    )
    cursor = conn.cursor()
    cursor.callproc('dbms_output.enable')
except oracledb.Error as e:
    print(e)

Modularización de Cálculos (Funciones):

Cree una función llamada salario_dep que reciba el ID de un departamento y retorne la suma total de salarios de todos los empleados que pertenecen a él. Si el departamento no tiene empleados, debe retornar 0.

Cree una función llamada compa_dep que reciba el ID de un departamento y calcule cuántos compañeros tiene un empleado en ese departamento (es decir, el conteo total de empleados del departamento menos uno). Asegúrese de que el resultado nunca sea negativo.

Procesamiento de Datos (Procedimiento):

Cree un procedimiento almacenado llamado proceso.

Dentro del procedimiento, declare un Cursor Explícito que obtenga el ID del empleado, el nombre del departamento y el ID del departamento, realizando un JOIN entre las tablas employees y departments.

Utilice un ciclo de control LOOP manual (con OPEN, FETCH y CLOSE) para recorrer el cursor. No se permite el uso de FOR LOOP automático para esta sección.

Para cada fila recuperada, imprima en consola (vía DBMS_OUTPUT) una línea con el siguiente formato: Emp: [ID] | Dept: [Nombre] | Compañeros: [Resultado Función 2] | Total Dept: [Resultado Función 1]

In [21]:
try:
    cursor.execute(
    """
    CREATE OR REPLACE FUNCTION salario_dep
    (p_dep_id IN departments.department_id%TYPE)
    RETURN NUMBER
    IS
        v_suma_salarios NUMBER;
    BEGIN
        SELECT NVL(SUM(salary), 0)
        INTO v_suma_salarios
        FROM employees 
        WHERE department_id = p_dep_id;
        RETURN v_suma_salarios;
    END salario_dep;
    """
    )

    cursor.execute(
    """
    CREATE OR REPLACE FUNCTION compa_dep
    (p_dep_id IN departments.department_id%TYPE)
    RETURN NUMBER
    IS
        v_compas NUMBER :=0;
    BEGIN
        SELECT COUNT(employee_id) - 1 
        INTO v_compas
        FROM employees
        WHERE department_id = p_dep_id;
        IF v_compas < 0 THEN
            v_compas := 0;
        END IF;
        RETURN v_compas;
    END compa_dep;
    """
    )

    cursor.execute(
    """
    CREATE OR REPLACE PROCEDURE pro_compas_salarios
    (p_limite IN NUMBER)
    IS
        CURSOR empleados IS
            SELECT e.first_name || ' ' || e.last_name, compa_dep(e.department_id), d.department_name, salario_dep(e.department_id)
            FROM employees e
            LEFT JOIN departments d ON e.department_id = d.department_id;
        v_nombre VARCHAR2(200);
        v_compas NUMBER;
        v_depa departments.department_name%TYPE;
        v_salario NUMBER;
        v_limite NUMBER := p_limite;
    BEGIN
        OPEN empleados;
        LOOP
            FETCH empleados INTO v_nombre, v_compas, v_depa, v_salario;
            EXIT WHEN empleados%NOTFOUND OR v_limite <= 0;
            DBMS_OUTPUT.PUT_LINE('Nombre: ' || v_nombre || ', Compas: ' || TO_CHAR(v_compas) || ', Depa: ' || v_depa || ', Salario: ' || TO_CHAR(v_salario));
            v_limite := v_limite-1;
        END LOOP;
        CLOSE empleados;
    END pro_compas_salarios;
    """
    )

    cursor.execute(
    """
    BEGIN
        pro_compas_salarios(15);
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc('dbms_output.get_line', (line, status))
        if(status.getvalue()!=0):
            break
        print(line.getvalue())
except oracledb.Error as e:
    print(e)

Nombre: Jennifer Whalen, Compas: 0, Depa: Administration, Salario: 4400
Nombre: Michael Hartstein, Compas: 1, Depa: Marketing, Salario: 19000
Nombre: Pat Fay, Compas: 1, Depa: Marketing, Salario: 19000
Nombre: Den Raphaely, Compas: 5, Depa: Purchasing, Salario: 24900
Nombre: Alexander Khoo, Compas: 5, Depa: Purchasing, Salario: 24900
Nombre: Shelli Baida, Compas: 5, Depa: Purchasing, Salario: 24900
Nombre: Sigal Tobias, Compas: 5, Depa: Purchasing, Salario: 24900
Nombre: Guy Himuro, Compas: 5, Depa: Purchasing, Salario: 24900
Nombre: Karen Colmenares, Compas: 5, Depa: Purchasing, Salario: 24900
Nombre: Susan Mavris, Compas: 0, Depa: Human Resources, Salario: 6500
Nombre: Matthew Weiss, Compas: 6, Depa: Shipping, Salario: 41200
Nombre: Adam Fripp, Compas: 6, Depa: Shipping, Salario: 41200
Nombre: Payam Kaufling, Compas: 6, Depa: Shipping, Salario: 41200
Nombre: Shanta Vollman, Compas: 6, Depa: Shipping, Salario: 41200
Nombre: Irene Mikkilineni, Compas: 6, Depa: Shipping, Salario: 41200


In [2]:
import oracledb
import pandas as pd
try:
    conn = oracledb.connect(
        user = 'hr',
        password = 'basededatos',
        dsn = 'localhost:1521/XE'
    )
    cursor = conn.cursor()
    cursor.callproc('dbms_output.enable')
except oracledb.Error as e:
    print(e)

listar los empleados, numero de companieros, y comprobar por cada empleado si el salario esta en los rangos

In [4]:
try:
    cursor.execute(
    """
    CREATE OR REPLACE FUNCTION f_salario
    (p_emp_id IN employees.employee_id%TYPE)
    RETURN VARCHAR2
    IS
        v_mensaje VARCHAR2(200);
        v_salario NUMBER;
    BEGIN
        SELECT salary
        INTO v_salario
        FROM employees
        WHERE employee_id = p_emp_id;
        v_mensaje := 
            CASE 
                WHEN v_salario < 1500 THEN 'Subir'
                WHEN v_salario BETWEEN 1500 AND 2500 THEN 'Salario medio'
                ELSE 'No subir'
            END;
        RETURN v_mensaje;
    EXCEPTION
        WHEN NO_DATA_FOUND THEN
            RETURN 'No tiene salario';
    END f_salario;
    """
    )

    cursor.execute(
    """
    CREATE OR REPLACE PROCEDURE pro_mensaje
    IS
        CURSOR empleados IS
            SELECT first_name || ' ' || last_name AS nombre, compa_dep(department_id) AS compas, f_salario(employee_id) AS mensaje
            FROM employees;
    BEGIN
        FOR empleado IN empleados LOOP
            DBMS_OUTPUT.PUT_LINE('Nombre: ' || empleado.nombre || ', Compas: ' || TO_CHAR(empleado.compas) || ', Mensaje: ' || empleado.mensaje);
        END LOOP;
    END pro_mensaje;
    """
    )

    cursor.execute(
    """
    BEGIN
        pro_mensaje;
    END;
    """
    )

    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc('dbms_output.get_line', (line,status))
        if(status.getvalue()!=0):
            break
        print(line.getvalue())
except oracledb.Error as e:
    print(e)

Nombre: Steven King, Compas: 2, Mensaje: No subir
Nombre: Neena Kochhar, Compas: 2, Mensaje: No subir
Nombre: Lex De Haan, Compas: 2, Mensaje: No subir
Nombre: Alexander Hunold, Compas: 4, Mensaje: No subir
Nombre: Bruce Ernst, Compas: 4, Mensaje: No subir
Nombre: David Austin, Compas: 4, Mensaje: No subir
Nombre: Valli Pataballa, Compas: 4, Mensaje: No subir
Nombre: Diana Lorentz, Compas: 4, Mensaje: No subir
Nombre: Nancy Greenberg, Compas: 5, Mensaje: No subir
Nombre: Daniel Faviet, Compas: 5, Mensaje: No subir
Nombre: John Chen, Compas: 5, Mensaje: No subir
Nombre: Ismael Sciarra, Compas: 5, Mensaje: No subir
Nombre: Jose Manuel Urman, Compas: 5, Mensaje: No subir
Nombre: Luis Popp, Compas: 5, Mensaje: No subir
Nombre: Den Raphaely, Compas: 5, Mensaje: No subir
Nombre: Alexander Khoo, Compas: 5, Mensaje: No subir
Nombre: Shelli Baida, Compas: 5, Mensaje: No subir
Nombre: Sigal Tobias, Compas: 5, Mensaje: No subir
Nombre: Guy Himuro, Compas: 5, Mensaje: No subir
Nombre: Karen Colmen

In [3]:
import oracledb
import pandas as pd
try:
    conn = oracledb.connect(
        user = 'hr',
        password = 'basededatos',
        dsn = 'localhost:1521/XE'
    )
    cursor = conn.cursor()
    cursor.callproc('dbms_output.enable')
except oracledb.Error as e:
    print(e)

dado un codigo de empleado cambiar su salario en 100
guardar en una tabla de auditoria el usuario, hora y empleado

In [7]:
try:
    cursor.execute(
    """
    DROP TABLE auditoria
    """
    )

    cursor.execute(
    """
    CREATE TABLE auditoria(
        usuario VARCHAR2(50),
        fecha DATE,
        empleado VARCHAR2(50)
    )
    """
    )
    cursor.execute(
    """
    BEGIN 
        DBMS_OUTPUT.PUT_LINE('Tabla creada con exito');
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc('dbms_output.get_line', (line,status))
        if(status.getvalue()!=0):
            break
        print(line.getvalue())
    
except oracledb.Error as e:
    print(e)
    cursor.execute(
    """
    CREATE TABLE auditoria(
        usuario VARCHAR2(50),
        fecha DATE,
        empleado VARCHAR2(50)
    )
    """
    )
    cursor.execute(
    """
    BEGIN 
        DBMS_OUTPUT.PUT_LINE('Tabla creada con exito');
    END;
    """
    )
    line = cursor.var(str)
    status = cursor.var(int)
    while True:
        cursor.callproc('dbms_output.get_line', (line,status))
        if(status.getvalue()!=0):
            break
        print(line.getvalue())
    

Tabla creada con exito


In [ ]:
try:
    cursor.execute(
    """
    ALTER TABLE auditoria ADD hora 
    """
    )
except oracledb.Error as e:
    print(e)

In [1]:
cursor.execute("""  
CREATE OR REPLACE TRIGGER trig
BEFORE UPDATE OF salary ON employees
FOR EACH ROW
BEGIN 
    INSERT INTO auditoria(usuario,hora,id_empleado)
    VALUES(USER, TO_CHAR(CURRENT_DATE, 'HH24:MI:SS'),:OLD.employee_id);
EXCEPTION
    WHEN NO_DATA_FOUND THEN
    DBMS_OUTPUT.PUT_LINE('EMPLEADO NO ENCONTRADO');
    WHEN OTHERS THEN 
    DBMS_OUTPUT.PUT_LINE('EL ERROR ES: '|| SQLCODE || ': '|| SQLERRM);
END; """)

cursor.execute("SELECT object_name, status FROM user_objects WHERE object_name= 'TRIG'")
obj= cursor.fetchone()
if obj:
    print(f"EL TRIGGER {obj[0]} ESTA {obj[1]}")
else:
    print("NO FUNCIONA EL TRIGGER")
    
cursor.execute("""  BEGIN
    UPDATE employees
    SET salary=5000
    WHERE employee_id=101;
    
    IF SQL%ROWCOUNT =0 THEN 
        DBMS_OUTPUT.PUT_LINE('NO SE HA ACTUALIZADO NADA (NO EXISTE EL USUARIO');
    ELSE
        DBMS_OUTPUT.PUT_LINE('ACTUALIZADO Y EN AUDITORIA');
    END IF;
END;""")


cursor.execute("""  SELECT * FROM auditoria """)

resultados= cursor.fetchall()
columnas= ["USUARIO", "HORA", "ID"]
df= pd.DataFrame(resultados, columns=columnas)
print(df)

NameError: name 'cursor' is not defined